题目来源: [LeetCode 3551](https://leetcode.cn/problems/minimum-swaps-to-sort-by-digit-sum/description/)

给你一个由 互不相同 的正整数组成的数组 nums，需要根据每个数字的数位和（即每一位数字相加求和）按 升序 对数组进行排序。
如果两个数字的数位和相等，则较小的数字排在前面。
返回将 nums 排列为上述排序顺序所需的 最小 交换次数。
一次 交换 定义为交换数组中两个不同位置的值。

上面的问题可以拆解成两部分

## Part1: 排序
将原始数组 nums 按照某一标准排序, 成 nums_sorted
    需要注意, 此处的排序是两个标准:
    - 数位和最小的排在前面
    - 数位和相同, 按照原始大小进行排序.
    
    因此, 可以把 sort()中的排序函数 key 定义成元组(digits_sum(num), num)
    因为元组会按照元素依次进行排序. 如元组长成(a, b, c), 会依次比较 a, b, c

    该部分由于生成了新的 nums_sorted, 空间复杂度 O(n); 时间复杂度 O(n * logn)

## Part2: 计算最小交换次数
统计只使用交换 nums 中两元素的方式, 将 nums 变成 nums_sorted的最小交换次数
很容易直接联想到线性代数中一个排列的'逆序'/'逆序数的概念(行列式部分会讲). 
但实际上, 逆序数 和 上面要求的最小交换次数, 没啥关系, 都没有相同的奇偶性.


与这道题要求的交换词数相关的概念来自抽象代数中的**置换/交错群**.
核心来自置换群理论--把从原数组 `nums` 变换到目标数组 `nums_sorted` 的过程看作一个**置换 (\sigma)**：

  - 对位置 i：原数组位置 i 的元素 (nums[i])，在目标数组中应当放到 `nums_sorted` 所在的位置。
  - 置换可以唯一分解成若干互不相交的**轮换（循环）**。
  - 若某个轮换长度为 s（轮换内包含 s 个元素），则把这个轮换调整到位，最少需要 (s-1) 次交换。
  - 总最小交换次数 = (\sum (s_i - 1))，其中 (s_i) 是第 i 个轮换的长度。

如, nums=[11, 23, 14, 6], 则按照上述规则完成排序后, 
nums_sorted=[11, 14, 23, 6],
nums -> nums_sorted的变换 可以写成 (11)(23 14)(6)这种形式--置换的轮换表示法.
每个括号中称之为一个轮换.

又比如, nums=[1, 4, 2, 8, 5, 7], nums_sorted=[1, 2, 4, 5, 7, 8]
nums -> nums_sorted的变换 可以写成$(1)(4 2)(8 5 7)$
  - 轮换 (1)：长度 (s=1)，贡献 (1-1=0)
  - 轮换 (4,2)：长度 (s=2)，贡献 (2-1=1)
  - 轮换 (8,5,7)：长度 (s=3)，贡献 (3-1=2)
  总交换次数 (0+1+2=3)。
      
可以观察出来, 假设 nums->nums_sorted 可以拆成 $k_1, k_2, ... k_m$ 个括号(每个括号都不能再拆开)
$k_i$ 括号中有 $s_i$ 个元素, 那么最小交换次数为:
$$\sum_{1\leq i \leq m}{(s_i-1)}$$

更多内容详见 石生明<近世代数初步(第二版)>P11.

所以问题也就变成了求 $k_i$. 如果 $k_i$ 以列表/集合的形式表现出来, 那 $s_i=len(k_i)$.
求 $k_i$ 的方法, 则需要借助图论中的连通的概念.
以上面的(1)(4 2)(8 5 7), 每个括号内部实际是封闭的:括号内部的元素可以任意连接交换, 但无法得到括号外面的元素.
所以, 每个括号实际是一个**连通分量**.

建图方式如下: (时间复杂度 O(n))
  1. 图的顶点：数组里的元素（因为所有数字互不相同，可以直接作为唯一节点标识）
  2. 建边规则：对每个下标 i，在顶点 `nums[i]` 和顶点 `nums_sorted[i]` 之间连一条无向边。
  3. 建好图之后，每一个连通分量，恰好对应置换里的一个轮换。连通分量大小 s 就是轮换长度。
  4. 遍历所有连通分量，累加 `s - 1`，得到答案。
  求出这个无向图的连通分量即可.

求连通分量的方法, 可以使用 **DFS/ BFS**. 下面采用 BFS. 复杂度 O(|V|+|E|), 
其中|V|=n, |E|<=n. 因此, 求连通分量这一步复杂度 O(n).
**但其实, 如果当置换环长度很大(例如一条长链环 (0 1 2 ... n-1), 连通分量很大, DFS递归调用次数过多,直接触发 `RecursionError`. 因此推荐 BFS.**
需要通过集合/数组来保存连通分量, 并在完成一个连通变量的遍历后, 存储当前连通分量. 详见下方代码.
[注: 同目录下 `强连通分量SCC`文件夹中有 有向图求强连通分量的代码实现, 但无向图在这里和之差不太多, 可以直接套用]

In [ ]:
def minSwaps(nums: list[int]) -> int:

    n=len(nums)

    # 排序, 时间复杂度 O(n * logn), 空间复杂度 O(n)
    def digits_sum(num):
        ans=0
        while num>0:
            ans+=num%10
            num//=10
        return ans
    nums_sorted=sorted(nums, reverse=False, key=lambda x: (digits_sum(x), x))
    print(nums_sorted)

    # 建图, 时间复杂度O(n), 空间复杂度 O(n)
    inverse=dict()
    for i, num in enumerate(nums):
        inverse[num]=i
    edges=[set() for _ in range(n)]
    for i in range(n):
        j=inverse[nums_sorted[i]]
        if i!=j: # 避免出现自环
            edges[i].add(j)
            edges[j].add(i)

    # 求图中各个连通分量, BFS, 时间空间复杂度均为 O(n)
    visited=[False for _ in range(n)]
    connected_components=[]

    # BFS
    from collections import deque

    def _bfs(u, tmp_cc):
        que=deque()
        que.append(u)
        visited[u]=True
        while que:
            u=que.popleft()
            tmp_cc.append(u)
            for v in edges[u]:
                if not visited[v]:
                    visited[v]=True
                    que.append(v)

    for u in range(n):
        if not visited[u]:
            tmp_cc=[] # cc: connected_components
            _bfs(u, tmp_cc)
            if tmp_cc: # 那些tmp_cc==[]的, 不加入 联通分量中, 否则下面 sum 求和会算出负数
                connected_components.append(tmp_cc)
    return sum(len(cc)-1 for cc in connected_components)
        
        
# 总体来看, 时间复杂度 O(n * logn), 空间复杂度 O(n).


In [3]:
# 以下是一些测试样例:
nums1=[978678783,989995084,465932830,37255967]
nums2=[37, 100]
nums3=[22,14,33,7]
nums4=[18,43,34,16]
numss=[nums1, nums2, nums3, nums4]
for nums in numss:
    print(minSwaps(nums))

[465932830, 37255967, 989995084, 978678783]
3
[100, 37]
1
[22, 14, 33, 7]
0
[16, 34, 43, 18]
2


In [ ]:
# 下面是一段可以使用的用DFS求联通分量的代码, 在 DFS 递归深度不是很深的时候使用
# n, visited, edges, connected_components 定义见上
def find_connected_components_dfs(u:int, cc:list): 
        # cc: connected components
        if not visited[u]:
            visited[u]=True
            cc.append(u)
            for v in edges[u]:
                find_connected_components_dfs(v, cc)

for u in range(n):
    tmp_cc=[]
    find_connected_components_dfs(u, tmp_cc)
    if tmp_cc: # 那些tmp_cc==[]的, 不加入 联通分量中, 否则下面 sum 求和会算出负数
        connected_components.append(tmp_cc)
print(connected_components)
